## 1. Install Dependencies

In [13]:
# Install into the CURRENT notebook kernel
%pip install -U pip setuptools wheel
%pip cache purge


  Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Using cached wheel-0.46.3-py3-none-any.whl (30 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 65.5.0
    Uninstalling setuptools-65.5.0:
      Successfully uninstalled setuptools-65.5.0
  Attempting uninstall: pip
    Found existing installation: pip 22.3.1
    Uninstalling pip-22.3.1:
      Successfully uninstalled pip-22.3.1
Files removed: 1023 (11954.8 MB)
Directories removed: 777
Note: you may need to restart the kernel to use updated packages.


In [20]:
# tested on python 3.12.10
%pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu128
%pip install -U ultralytics roboflow ncnn==1.0.20260114 pnnx==20260112


Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 10.2 MB/s  0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.6
    Uninstalling ultralytics-8.4.6:
      Successfully uninstalled ultralytics-8.4.6
Note: you may need to restart the kernel to use updated packages.


## 2. Download and prepare dataset

In [21]:
from roboflow import Roboflow
rf = Roboflow(api_key="Mt0m4dCKTTlNf29OV3zm")
project = rf.workspace("dylans-workspace-3init").project("ping-pong-detection-0guzq-f7zly")
version = project.version(1)
dataset = version.download("yolo26") # This downloads the images AND the data.yaml
                

loading Roboflow workspace...
loading Roboflow project...


### 2.1 Patch dataset with missing labels

In [17]:
from pathlib import Path
import re
import yaml

EXCLUDED_DIR_PARTS = {"venv", ".git", "node_modules", "build", "ultralytics-src"}

COCO_NAMES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
    "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
    "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors", "teddy bear",
    "hair drier", "toothbrush",
]

def is_excluded(path: Path) -> bool:
    return any(part in EXCLUDED_DIR_PARTS for part in path.parts)

def resolve_dataset_root() -> Path:
    # 1) Preferred: use Roboflow dataset object from the previous cell if present.
    ds = globals().get("dataset", None)
    if ds is not None:
        for attr in ("location", "dataset_path", "path"):
            value = getattr(ds, attr, None)
            if value:
                p = Path(value).resolve()
                if (p / "data.yaml").exists():
                    return p

    # 2) Common local folders used in this notebook.
    for name in ("My-First-Project-1", "Ping-Pong-Detection-1", "ping-pong-detection-1"):
        p = Path(name).resolve()
        if (p / "data.yaml").exists():
            return p

    # 3) Fallback: scan workspace for a dataset-like data.yaml.
    cwd = Path.cwd().resolve()
    candidates = []
    for data_yaml in cwd.rglob("data.yaml"):
        root = data_yaml.parent
        if is_excluded(root):
            continue
        has_split_dirs = (root / "train").exists() and (root / "valid").exists()
        score = 1 if has_split_dirs else 0
        candidates.append((score, root))

    if candidates:
        candidates.sort(key=lambda item: item[0], reverse=True)
        return candidates[0][1]

    raise FileNotFoundError(
        "Could not locate dataset data.yaml. Run the Roboflow download cell first and ensure it creates a folder with data.yaml. "
        f"Current working directory: {cwd}"
    )

dataset_root = resolve_dataset_root()
print(f"Using dataset root: {dataset_root}")

def rewrite_data_yaml(dry_run: bool = False):
    path = dataset_root / "data.yaml"
    data = yaml.safe_load(path.read_text(encoding="utf-8"))

    original_names = data.get("names", [])
    print(f"Original names: {original_names}")

    # Force dataset to YOLO COCO label space so class id 32 is 'sports ball'.
    data["names"] = COCO_NAMES
    data["nc"] = len(COCO_NAMES)

    if dry_run:
        print("Dry run: data.yaml would be rewritten with COCO names.")
        return

    path.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")
    print(f"Updated: {path}")

def rewrite_label_file(path: Path, target_class: int, dry_run: bool = False):
    changed_lines = 0
    original = path.read_text(encoding="utf-8").splitlines(keepends=True)
    updated = []

    for line in original:
        stripped = line.strip()

        if not stripped:
            updated.append(line)
            continue

        parts = stripped.split(maxsplit=1)
        first = parts[0]
        rest = parts[1] if len(parts) > 1 else ""

        # Accept integer-like or float-like ids and normalize to class 32.
        if not re.fullmatch(r"[-+]?\d+(?:\.\d+)?", first):
            updated.append(line)
            continue

        new_line = f"{target_class} {rest}".rstrip()
        if line.endswith("\n"):
            new_line += "\n"

        if new_line != line:
            changed_lines += 1
        updated.append(new_line)

    if changed_lines and not dry_run:
        path.write_text("".join(updated), encoding="utf-8")

    return changed_lines

def change_label_classes(dry_run: bool = True):
    target_class = 32
    label_dirs = [
        dataset_root / "train" / "labels",
        dataset_root / "valid" / "labels",
        dataset_root / "test" / "labels",
    ]

    total_files = 0
    touched_files = 0
    total_lines_changed = 0

    for label_dir in label_dirs:
        if not label_dir.exists():
            print(f"Skipping missing directory: {label_dir}")
            continue

        for txt_file in label_dir.rglob("*.txt"):
            total_files += 1
            changed = rewrite_label_file(txt_file, target_class, dry_run=dry_run)
            if changed:
                touched_files += 1
                total_lines_changed += changed
                print(f"Updated {txt_file} ({changed} line(s))")

    mode = "DRY RUN" if dry_run else "WRITE"
    print(f"\n[{mode}] Scanned files: {total_files}")
    print(f"[{mode}] Files changed: {touched_files}")
    print(f"[{mode}] Label lines changed: {total_lines_changed}")


dry_run = False  # Set to True to preview only
rewrite_data_yaml(dry_run=dry_run)
change_label_classes(dry_run=dry_run)

Using dataset root: C:\YOLO-android\ncnn\ncnn-android-yolo11\Ping-Pong-Detection-1
Original names: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
Updated: C:\YOLO-android\ncnn\ncnn-andr

# 3. Train

In [18]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3060


In [ ]:
import sys
import subprocess
from pathlib import Path

try:
    from ultralytics import YOLO, settings
except ModuleNotFoundError:
    print("ultralytics not found in this kernel. Installing editable package from ./ultralytics-src ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "./ultralytics-src"])
    from ultralytics import YOLO, settings

settings.reset()  # reset output dirs if different envs messed with it

# Reuse dataset_root from Cell 7 if available; fallback to common path.
data_yaml = Path("My-First-Project-1") / "data.yaml"
if "dataset_root" in globals():
    data_yaml = Path(dataset_root) / "data.yaml"

if not data_yaml.exists():
    raise FileNotFoundError(f"data.yaml not found at: {data_yaml}. Run dataset download + patch cells first.")

# 1. Load a pretrained YOLO26n model
model = YOLO("yolo26n.pt")

# 2. Train the model
model.train(data=str(data_yaml), epochs=100, imgsz=640, batch=32)

ultralytics not found in this kernel. Installing editable package from ./ultralytics-src ...


ModuleNotFoundError: No module named 'ultralytics'

## 3. Export YOLO26 NCNN

In [1]:
train_dir = "runs/detect/train/weights"

In [2]:
from pathlib import Path
from ultralytics import YOLO
YOLO(Path(train_dir)/"best.pt").export(**{
    'format': 'ncnn',
    'opset': 20,
    'simplify': True,
    'batch': 1,
    'imgsz': 640,
})

Ultralytics 8.4.6  Python-3.12.10 torch-2.10.0+cu128 CPU (AMD Ryzen 7 5800H with Radeon Graphics)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs

PyTorch: starting from 'runs\detect\train\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8400, 84) (5.3 MB)

NCNN: starting export with NCNN 1.0.20260114 and PNNX 20260112...
NCNN: export success  22.6s, saved as 'runs\detect\train\weights\best_ncnn_model' (4.7 MB)

Export complete (23.1s)
Results saved to C:\Users\thesp\Documents\Projects\Unibots\BARF\runs\detect\train\weights
Predict:         yolo predict task=detect model=runs\detect\train\weights\best_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=runs\detect\train\weights\best_ncnn_model imgsz=640 data=My-First-Project-1/data.yaml  
Visualize:       https://netron.app


'runs\\detect\\train\\weights\\best_ncnn_model'

### 3.1 Copy the exported model to app assets

In [3]:
import shutil
from pathlib import Path

assets_dir = Path("app") / "src" / "main" / "assets"
renames = {
    "model.ncnn.bin": "yolo26n.ncnn.bin",
    "model.ncnn.param": "yolo26n.ncnn.param",
}
for src, dst in renames.items():
    shutil.copy((Path(train_dir)/"best_ncnn_model"/src).resolve(), (assets_dir / dst).resolve())
    print(f"Copied {src} -> {assets_dir / dst}")

Copied model.ncnn.bin -> app\src\main\assets\yolo26n.ncnn.bin
Copied model.ncnn.param -> app\src\main\assets\yolo26n.ncnn.param
